# PHYS 132 Activity 1: Pressure, radiation and opacity in a model of the Sun

We use a standard solar model, BS05(OP) of Bahcall, Serenelli and Basu (2005, astro-ph/0412440), which is a
numerical solution of the equations of stellar structure for the Sun as it is today, tabulated in 1268 shells
from the centre out to $0.983\,R_\odot$. For each shell the table gives the mass $m$ inside it, its radius $r$,
the temperature $T$, density $\rho$ and pressure $P$, the luminosity $L$ flowing out through it, and the mass
fractions of hydrogen ($X$), helium ($Y$) and a few heavier nuclei. Everything is in cgs units, as in the
lecture notes.

Run the first cell as it is. In each part after that, fill in the lines marked `...` and run the cell. The
questions at the end of each part are for your group to discuss and to answer in a sentence or two.

In [ ]:
# Run this cell first. It loads the solar model and defines the constants we need, in cgs units.
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid, trapezoid
np.seterr(divide="ignore", invalid="ignore")   # the innermost shells have L = 0 and repeated values

URL = "https://www.sns.ias.edu/~jnb/SNdata/Export/BS2005/bs05op.dat"
if not os.path.exists("bs05op.dat"):
    # If this download fails, get bs05op.dat from Canvas and upload it next to this notebook.
    urllib.request.urlretrieve(URL, "bs05op.dat")

rows = []
for line in open("bs05op.dat"):
    words = line.split()
    if len(words) == 12 and words[0][0].isdigit():
        rows.append([float(w) for w in words])
table = np.array(rows)

G    = 6.674e-8      # gravitational constant, cm^3 g^-1 s^-2
a    = 7.566e-15     # radiation constant, erg cm^-3 K^-4
c    = 2.998e10      # speed of light, cm s^-1
k_B  = 1.381e-16     # Boltzmann constant, erg K^-1
m_u  = 1.661e-24     # atomic mass unit, g
Msun = 1.989e33      # g
Rsun = 6.9598e10     # cm     (the solar radius and luminosity used to build the model)
Lsun = 3.8418e33     # erg s^-1

m   = table[:, 0] * Msun    # mass inside the shell, g
r   = table[:, 1] * Rsun    # radius of the shell, cm
T   = table[:, 2]           # temperature, K
rho = table[:, 3]           # density, g cm^-3
P   = table[:, 4]           # pressure, dyn cm^-2
L   = table[:, 5] * Lsun    # luminosity flowing out through the shell, erg s^-1
X   = table[:, 6]           # hydrogen mass fraction
Y   = table[:, 7]           # helium-4 mass fraction

print(f"{len(r)} shells, from r = {r[0]/Rsun:.4f} R_sun to r = {r[-1]/Rsun:.3f} R_sun")

## The model at a glance

The top row shows temperature, density and pressure against radius, with logarithmic vertical axes; the bottom
row shows the mass inside each radius, the luminosity flowing out through it, and the mass fractions of hydrogen
and helium. Look at each panel before going on.

1. Read off how far $T$, $\rho$ and $P$ fall between the centre and the outermost shell. The pressure falls by
   much more than either of the other two; why should that be, given that the gas is ideal?
2. How far out do you have to go to enclose half the mass? Half the luminosity?
3. What has happened to the composition at the centre, and why? What is unusual about it outside about
   $0.7\,R_\odot$? Keep that in mind for part 3.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, y, label in zip(axes[0], [T, rho, P],
                        [r"$T$ (K)", r"$\rho$ (g cm$^{-3}$)", r"$P$ (dyn cm$^{-2}$)"]):
    ax.semilogy(r / Rsun, y)
    ax.set_ylabel(label)
axes[1, 0].plot(r / Rsun, m / Msun)
axes[1, 0].set_ylabel(r"$m/M_\odot$")
axes[1, 1].plot(r / Rsun, L / Lsun)
axes[1, 1].set_ylabel(r"$L/L_\odot$")
axes[1, 2].plot(r / Rsun, X, label="hydrogen, $X$")
axes[1, 2].plot(r / Rsun, Y, label="helium, $Y$")
axes[1, 2].set_ylabel("mass fraction")
axes[1, 2].legend()
for ax in axes.flat:
    ax.set_xlabel(r"$r/R_\odot$")
plt.tight_layout()
plt.show()

print(f"T falls by a factor {T[0]/T[-1]:.3g}, rho by {rho[0]/rho[-1]:.3g}, P by {P[0]/P[-1]:.3g}")

## 1. Hydrostatic equilibrium

The model was constructed to satisfy hydrostatic equilibrium,
$$
\frac{dP}{dr} = -\frac{G m \rho}{r^2},
$$
so we can check it using nothing but the columns $m$, $\rho$ and $r$. Start from the pressure in the outermost
shell and integrate inward,
$$
P(r) = P(r_{\rm out}) + \int_r^{r_{\rm out}} \frac{G m \rho}{r'^2}\,dr',
$$
and compare the result with the tabulated pressure. The function `cumulative_trapezoid(y, x, initial=0)`
returns the running integral $\int_{x_0}^{x} y\,dx'$ at every point of `x`, and the cell turns that into the
integral from $r$ outward. You only need to write the right-hand side of hydrostatic equilibrium.

In [ ]:
dPdr = ...   # the right-hand side of hydrostatic equilibrium, one value per shell

running = cumulative_trapezoid(-dPdr, r, initial=0)   # integral of G m rho / r^2 from the centre to r
P_hse = P[-1] + (running[-1] - running)               # integral from r to the last shell, plus P there

print(f"central pressure in the table:            {P[0]:.4e} dyn/cm^2")
print(f"central pressure integrated from outside: {P_hse[0]:.4e} dyn/cm^2")

plt.plot(r / Rsun, P_hse / P - 1)
plt.xlabel(r"$r/R_\odot$")
plt.ylabel(r"$P_{\rm HSE}/P - 1$")
plt.show()

How well does the model satisfy hydrostatic equilibrium? In Lecture 2 we estimated
$P_c \sim GM_\odot^2/R_\odot^4 \approx 10^{16}$ dyn cm$^{-2}$ and proved the bound
$P_c \ge 4\times10^{14}$ dyn cm$^{-2}$. How do both compare with the model?

## 2. Radiation pressure

Blackbody radiation at temperature $T$ exerts a pressure $P_{\rm rad} = aT^4/3$, which we derived in the
lecture. In Lecture 2 we took the pressure to be that of an ideal gas and ignored the radiation. Compute the
fraction of the pressure due to radiation in every shell.

In [ ]:
P_rad = ...   # radiation pressure in each shell

frac = P_rad / P
i = np.argmax(frac)
print(f"at the centre, P_rad/P = {frac[0]:.2e}")
print(f"largest value, P_rad/P = {frac[i]:.2e}, at r = {r[i]/Rsun:.2f} R_sun")

plt.semilogy(r / Rsun, frac)
plt.xlabel(r"$r/R_\odot$")
plt.ylabel(r"$P_{\rm rad}/P$")
plt.show()

Was it safe to ignore radiation pressure in the Sun? The ratio goes as $T^4/P$, and we will come back to
it when we ask how massive a star can be.

## 3. Opacity

The transport equation from the lecture,
$$
\frac{dT}{dm} = -\frac{3\kappa L}{64\pi^2 a c\, r^4 T^3},
$$
can be read the other way round. The model gives $T$, $L$, $r$ and $m$, so it tells us the opacity $\kappa$
the gas must have for radiative diffusion to carry the luminosity. Solve for $\kappa$ and compute it in every
shell.

For comparison, the lecture showed that if scattering by free electrons were the only process, the opacity of
fully ionised gas would be $\kappa_{\rm es} = 0.20\,(1+X)$ cm$^2$ g$^{-1}$.

In [ ]:
dTdm = np.gradient(T, m)   # numerical derivative of T with respect to m
kappa = ...                # solve the transport equation for kappa
kappa_es = 0.20 * (1 + X)

for x in [0.1, 0.3, 0.5]:
    j = np.argmin(abs(r / Rsun - x))
    print(f"r = {x:.1f} R_sun: kappa = {kappa[j]:.2f} cm^2/g, kappa/kappa_es = {kappa[j]/kappa_es[j]:.1f}")

plt.semilogy(r / Rsun, kappa, ".", ms=2, label=r"$\kappa$ inferred from the model")
plt.semilogy(r / Rsun, kappa_es, label=r"electron scattering, $\kappa_{\rm es}$")
plt.ylim(0.1, 100)
plt.xlabel(r"$r/R_\odot$")
plt.ylabel(r"$\kappa$ (cm$^2$ g$^{-1}$)")
plt.legend()
plt.show()

The mean free path of a photon is $\lambda = 1/\kappa\rho$. Compute it from the inferred opacity and plot it;
only the region inside the break is meaningful, for reasons the next questions get at.

In [ ]:
lam = ...   # photon mean free path in each shell, cm

inside = r < 0.7 * Rsun
plt.semilogy(r[inside] / Rsun, lam[inside], ".", ms=2)
plt.xlabel(r"$r/R_\odot$")
plt.ylabel(r"$\lambda$ (cm)")
plt.show()

1. How much larger than $\kappa_{\rm es}$ is the inferred opacity near the centre, and at $0.5\,R_\odot$?
   Something other than electron scattering must be absorbing the photons; we meet those processes later in
   the course.
2. The curve turns over abruptly near $0.7\,R_\odot$; find where. The real opacity of the gas does not turn over
   there: it keeps rising outward (we will see why in November). With the real opacity, radiation alone would
   need a steeper temperature gradient than the model has beyond this radius. What does that say about how the
   energy is carried there? Compare with the composition panel at the top of the notebook. Keep the radius you
   found; we will predict it from first principles in November.
3. Why is the curve noisy close to the centre? (Look at what happens to $L$ and to $dT/dm$ as $r \to 0$, and
   remember that the table gives $T$ to four significant figures.)